In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from analysis_utils.utils import get_things, run_for_n_iters, get_things_wandb, get_model_epochs
import matplotlib.pyplot as plt
import torch
import numpy as np
import random

In [ ]:
import experts_initializers as exps

runs = {
    "spatial_vertical": "zzy1ob3e",
    #"spatial_horizontal": "",
    "labels": "cnmg4o00",
    "temporal": "2v7m2j61",
    #"spatial_center": ""
}

routers = {
    "spatial_vertical": exps.SpatialRouterVertical(3, 14, 14, 8),
    "spatial_horizontal": exps.SpatialRouterHorizontal(3, 14, 14, 8),
    "labels": exps.TokensToLabels(3),
    "temporal": exps.TemporalRouter(3, 14, 14, 8),
    "spatial_center": exps.SpatialRouterCentral(2, 14, 14, 8)
}

In [ ]:
SEED = 0
torch.manual_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

In [ ]:
models_outputs = {}
n_iters = 3
device = 'cuda'
batch_size = 25

for model_name, RUN_ID in runs.items():
    config, run_model_name = get_things_wandb(RUN_ID)
    epochs = get_model_epochs(run_model_name)
    models_outputs[model_name] = {}
    for e, epoch in enumerate(epochs):
        print(f"Running {model_name}, Epoch: {epoch}, Progress {e+1}/{len(epochs)}")
        model, training_loader, validation_loader, test_loader, config = get_things(config, 25, SEED, epoch, run_model_name)
        results = run_for_n_iters(n_iters, model, config, validation_loader, device)
        models_outputs[model_name][epoch] = results

In [ ]:
models_epochs_accuracies = {}

for model in models_outputs:
    models_epochs_accuracies[model] = {}
    router = routers[model_name]
    for epoch in models_outputs[model]:
        models_epochs_accuracies[model][epoch] = {}
        for layer in models_outputs[model][epoch]:
            matches = 0
            total = 0
            for i, batch in enumerate(models_outputs[model][epoch][layer]["routes"]):
                labels = models_outputs[model][epoch][layer]["labels"][i]
                if model != "labels":
                    seq_len = int(len(batch) / batch_size)
                    good_routes = router.route(batch_size, seq_len, device).cpu().numpy()
                    matches += (good_routes == np.array(batch)).sum()
                    total += len(batch)
                elif model == "labels":
                    seq_len = len(batch)
                    good_routes = router.route(batch_size, seq_len, device, labels=labels).cpu().numpy()
                    matches += (good_routes == np.array(batch)).sum()
                    total += len(batch)                    
            models_epochs_accuracies[model][epoch][layer] = matches / total


In [ ]:
models_epochs_accuracies['temporal']